In [76]:
from elastica._calculus import _isnan_check
from elastica.timestepper import extend_stepper_interface
from elastica import *
from elastica._elastica_numba._rod._ribbon1D import Ribbon1D
from elastica._elastica_numba._rod._linear_ribbon1D import LinearRibbon1D

from Cases.arm_function import(
    DampingFilterBC,
    ExponentialDampingBC,
    DampingFilterBCRingRod,)

from elastica._linalg import _batch_norm

from Cases.post_processing import (plot_video_with_surface,plot_video_activation_muscle,)

import os
from elastica._rotations import _get_rotation_matrix

from itertools import groupby

from Connections import *

In [82]:
class RibbonSimulator_withOgden(BaseSystemCollection, Constraints, MemoryBlockConnections, Forcing, CallBacks):
    pass


ribbon_bollean = 1    
n_elem = 100
start = np.array([0.0, 0.0, 0.0])
direction = np.array([0.0, 0.0, 1.0])
normal = np.array([0.0, 1.0, 0.0])
base_length = 50
thickness = 0.1
width = 5.0
base_area = width*thickness
density = 1.017e-5
nu = 1e-5
E = 2.77e3
poisson_ratio = 0.34
shear_modulus = E / (poisson_ratio + 1.0)
#shear_modulus = E 

dl = base_length / n_elem
dt = 2.0e-6

origin_force = np.array([0.0, 0.0, 0.0])
end_force = np.array([0.0, -0.0092, 0.0])
ramp_up_time = 15.0



Ribbon_Ogden = RibbonSimulator_withOgden()


if ribbon_bollean==2:
    ribbon = Ribbon1D.straight_ribbon(
        n_elem,
        start,
        direction,
        normal,
        base_length,
        thickness,
        width,
        density,
        youngs_modulus=E,
        shear_modulus=shear_modulus,
        poisson_ratio = poisson_ratio,
        nu = nu,
    )
    Ribbon_Ogden.append(ribbon)

elif ribbon_bollean == 2:
    ribbon = LinearRibbon1D.straight_ribbon(
        n_elem,
        start,
        direction,
        normal,
        base_length,
        thickness,
        width,
        density,
        youngs_modulus=E,
        shear_modulus=shear_modulus,
        poisson_ratio = poisson_ratio,
        nu = nu,
    )
    Ribbon_Ogden.append(ribbon)

else:
    ribbon = CosseratRod.straight_rod(
        n_elem,
        start,
        direction,
        normal,
        base_length,
        thickness,
        density,
        youngs_modulus=E,
        shear_modulus=shear_modulus,
        poisson_ratio = poisson_ratio,
        nu = nu,
    )
    Ribbon_Ogden.append(ribbon)    


Ribbon_Ogden.constrain(ribbon).using(
    DampingFilterBC,
    constrained_position_idx=(0,),
    constrained_director_idx=(0,),
    filter_order=5,  # 10,
)


Ribbon_Ogden.constrain(ribbon).using(
    OneEndFixedRod, constrained_position_idx=(0,), constrained_director_idx=(0,)
)
Ribbon_Ogden.add_forcing_to(ribbon).using(
    EndpointForces, origin_force, end_force, ramp_up_time=ramp_up_time
)

gravitational_acc = -9.80665*0
Ribbon_Ogden.add_forcing_to(ribbon).using(
    GravityForces, acc_gravity=np.array([0.0, gravitational_acc, 0.0])
)



In [83]:
class RibbonOgdenCallBack(CallBackBaseClass):
    """
    Call back function for Bean Ogeden penetration
    """

    def __init__(self, step_skip: int, callback_params: dict):
        CallBackBaseClass.__init__(self)
        self.every = step_skip
        self.callback_params = callback_params

    def make_callback(self, system, time, current_step: int):

        if current_step % self.every == 0:

            self.callback_params["time"].append(time)
            self.callback_params["step"].append(current_step)
            self.callback_params["position"].append(system.position_collection.copy())
            self.callback_params["velocity"].append(system.velocity_collection.copy())
            self.callback_params["avg_velocity"].append(
                system.compute_velocity_center_of_mass()
            )

            self.callback_params["center_of_mass"].append(
                system.compute_position_center_of_mass()
            )
            self.callback_params["curvature"].append(system.kappa.copy())
            self.callback_params["sigma"].append(system.sigma.copy())
            self.callback_params["internal_stress"].append(system.internal_stress.copy())
            self.callback_params["internal_couple"].append(system.internal_couple.copy())

            return


pp_list = defaultdict(list)
Ribbon_Ogden.collect_diagnostics(ribbon).using(
    RibbonOgdenCallBack, step_skip=100000, callback_params=pp_list
)
print("Callback function added to the simulator")

Callback function added to the simulator


In [84]:
Ribbon_Ogden.finalize()
print("System finalized")

System finalized


In [85]:
final_time = 15.0
total_steps = int(final_time / dt)
print("Total steps to take", total_steps)

timestepper = PositionVerlet()

Total steps to take 7500000


In [86]:
integrate(timestepper, Ribbon_Ogden, final_time, total_steps)

positions_over_time = np.array(pp_list["position"])
if (np.isnan(positions_over_time)==False).all()==False:
    print("Simulation diverge. Try lowering time step !")

100%|██████████| 7500000/7500000 [33:33<00:00, 3724.04it/s]  

Final time of simulation is :  14.999999995525515


In [87]:
pp_list["curvature"][-1]

array([[-0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
        -0.00000000e+00, -0.00000000e+00, -0.00000000e+0

In [88]:
pp_list["internal_couple"][-1]

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+0

In [89]:
pp_list["internal_stress"][-1]

array([[-9.70102661e-03, -7.31705602e-03, -1.06346566e-02,
        -6.45431380e-03, -9.77355935e-03, -5.64902806e-03,
        -8.74575147e-03, -4.71416306e-03, -7.69404123e-03,
        -3.78717178e-03, -6.71797568e-03, -2.92719275e-03,
        -5.85972515e-03, -2.16032048e-03, -5.13030082e-03,
        -1.49264186e-03, -4.52408070e-03, -9.19829136e-04,
        -4.02790655e-03, -4.32890581e-04, -3.62615162e-03,
        -2.13633583e-05, -3.30332994e-03,  3.25051365e-04,
        -3.04529847e-03,  6.15742412e-04, -2.83969862e-03,
         8.58922793e-04, -2.67601053e-03,  1.06160767e-03,
        -2.54542650e-03,  1.22970521e-03, -2.44065288e-03,
         1.36815063e-03, -2.35569631e-03,  1.48104837e-03,
        -2.28566051e-03,  1.57180572e-03, -2.22656453e-03,
         1.64325126e-03, -2.17518580e-03,  1.69773667e-03,
        -2.12892721e-03,  1.73722226e-03, -2.08570586e-03,
         1.76334827e-03, -2.04386097e-03,  1.77749370e-03,
        -2.00207780e-03,  1.78082466e-03, -1.95932518e-0

In [93]:
pp_list["sigma"]

[array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [96]:
from IPython.display import Video
from tqdm import tqdm


def plot_video_2D(plot_params: dict, video_name="video.mp4", margin=0.2, fps=15, plan_y_pos = None):
    from matplotlib import pyplot as plt
    import matplotlib.animation as manimation

    t = np.array(plot_params["time"])
    positions_over_time = np.array(plot_params["position"])
    total_time = int(np.around(t[..., -1], 1))
    total_frames = fps * total_time
    step = round(len(t) / total_frames)

    print("creating video -- this can take a few minutes")
    FFMpegWriter = manimation.writers["ffmpeg"]
    metadata = dict(title="Movie Test", artist="Matplotlib", comment="Movie support!")
    writer = FFMpegWriter(fps=fps, metadata=metadata)

    fig = plt.figure()
    ax = fig.add_subplot(111)
    plt.axis("equal")
    if plan_y_pos!= None:
        plt.axhline(y = plan_y_pos, color = 'r', linestyle = '--', linewidth = 1) 
    rod_lines_2d = ax.plot(
        positions_over_time[0][2], positions_over_time[0][1], linewidth=1
    )[0]
    limite = np.max(positions_over_time[0])
    ax.set_xlim([0 - margin, limite + margin])
    ax.set_ylim([-limite - margin, limite+ margin])
    with writer.saving(fig, video_name, dpi=100):
        with plt.style.context("seaborn-v0_8-whitegrid"):
            for time in range(1, len(t)-1, step):
                rod_lines_2d.set_xdata(positions_over_time[time][2])
                rod_lines_2d.set_ydata(positions_over_time[time][1])

                writer.grab_frame()
    plt.close(fig)


filename_video = "Ogden_video.mp4"
plot_video_2D(pp_list, video_name=filename_video, margin=0.2, fps=10)

Video("Ogden_video.mp4")

creating video -- this can take a few minutes
